# POC Triage CHSA — Entraînement SFT (Qwen3-1.7B-Base) — KaggleVersion **Kaggle uniquement** (plus de branche Colab, pour éviter la confusion qu'on a eue sur les chemins de sortie).**Avant de commencer :**- Panneau de droite *Settings* > *Accelerator* > **GPU T4 x2**.- Panneau de droite *Settings* > *Internet* > **On** (vérification par téléphone requise une seule fois).- *Add Input* > ajoute ton Dataset (`Agent_dataset`), qui doit contenir à sa racine : `train_sft.py`, `template_triage.jinja`, `train.jsonl`, `validation.jsonl`.**Ordre à suivre :**0. Configuration (chemins + vérification que le bon `train_sft.py` est monté)1. Vérifier le GPU2. Installer les dépendances (versions épinglées + retrait de `torchao`)3. Vérifier que les fichiers requis sont présents4. **Smoke test d'abord** (5 steps) — et lire les lignes `[masquage]`5. Entraînement réel, seulement après un smoke test réussi6. Récupérer le résultat

## 0. Configuration (chemins Kaggle)

In [1]:
import os, glob

assert "KAGGLE_KERNEL_RUN_TYPE" in os.environ, (
    "Ce notebook est la version Kaggle uniquement. Si tu es sur Colab, utilise l'ancien "
    "notebook multi-plateforme, ou lance ce notebook depuis kaggle.com/code."
)

# Le Dataset est monté sous /kaggle/input/<nom>/ : on cherche train_sft.py plutôt que de
# deviner le nom du Dataset (Agent_dataset chez toi, mais peu importe le nom exact).
candidats = glob.glob("/kaggle/input/**/train_sft.py", recursive=True)
assert candidats, (
    "train_sft.py introuvable dans /kaggle/input. Vérifie que ton Dataset est bien ajouté "
    f"au notebook (Add Input). Contenu de /kaggle/input : {os.listdir('/kaggle/input')}"
)
assert len(candidats) == 1, (
    f"Plusieurs train_sft.py trouvés, à vérifier à la main : {candidats}"
)

SRC = os.path.dirname(candidats[0])
DATA = SRC   # train.jsonl / validation.jsonl sont dans le même Dataset
OUT = "/kaggle/working"

print("Fichiers sources :", SRC)
print("Sorties          :", OUT)

Fichiers sources : /kaggle/input/datasets/veroniqueleroux/agent-dataset
Sorties          : /kaggle/working


## 1. Vérifier le GPU

In [2]:
!nvidia-smi
import torch

print("\nCUDA disponible :", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Pas de GPU détecté — Settings > Accelerator > GPU T4 x2, puis relance cette cellule."
)

nom = torch.cuda.get_device_name(0)
capacite = torch.cuda.get_device_capability(0)
bf16_natif = capacite[0] >= 8   # Ampere ou plus récent
print(f"GPU : {nom} (capacité de calcul {capacite[0]}.{capacite[1]})")
print("bf16 natif :", "OUI" if bf16_natif else
      "NON (T4, P100...) — train_sft.py bascule automatiquement en fp16 dans ce cas.")

Thu Sep 24 20:12:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Installer les dépendancesVersions épinglées, testées en local (TRL 1.13.0 exige `transformers>=4.56.2`, `accelerate>=1.4.0`, `datasets>=4.7.0`).**`torchao` est désinstallé volontairement** : `peft` le détecte automatiquement s'il est présent et plante (`ImportError: Found an incompatible version of torchao`) même quand on ne s'en sert pas — pas besoin de lui pour un LoRA classique.

In [3]:
!pip install -q "trl==1.13.0" "transformers==5.17.0" "accelerate>=1.4.0" "datasets>=4.7.0" peft
!pip uninstall -y torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 79.0 MB/s eta 0:00:00:00:01


## 3. Vérifier les fichiers requisVérifie aussi que le `train_sft.py` monté contient bien le correctif `loss_type="nll"` (sans lui, `SFTTrainer` plante avec `AttributeError: 'functools.partial' object has no attribute '__func__'` — un bug de TRL avec le calcul de loss "chunké" par défaut, sans rapport avec tes données).

In [4]:
FICHIERS_REQUIS = ["train_sft.py", "template_triage.jinja", "train.jsonl", "validation.jsonl"]

manquants = [f for f in FICHIERS_REQUIS if not os.path.exists(os.path.join(SRC, f))]
assert not manquants, f"Fichiers manquants dans {SRC} : {manquants}. Contenu : {os.listdir(SRC)}"
print("Fichiers requis présents dans", SRC)

chemin_script = os.path.join(SRC, "train_sft.py")
contenu_script = open(chemin_script, encoding="utf-8").read()
assert 'loss_type="nll"' in contenu_script, (
    "Le correctif loss_type=\"nll\" n'est PAS présent dans ce train_sft.py — "
    "c'est encore l'ancienne version (ou une version mal éditée). "
    "Remonte une nouvelle version du Dataset avec le fichier corrigé, puis Restart Session."
)
print("Correctif loss_type=\"nll\" bien présent dans train_sft.py")

Fichiers requis présents dans /kaggle/input/datasets/veroniqueleroux/agent-dataset
Correctif loss_type="nll" bien présent dans train_sft.py


## 4. SMOKE TEST — à faire en premier, toujoursObjectif : vérifier que le script tourne de bout en bout, **PAS** d'obtenir un bon modèle. `--max-steps 5` s'arrête très vite.**Que vérifier dans la sortie (« sans erreur » ne suffit pas) :**1. Les lignes **`[masquage] tokens appris : N/M`** : N doit être nettement inférieur à M.2. Le texte affiché juste après doit contenir **uniquement les paroles de l'agent**, chaque tour finissant par `<|im_end|>`.3. Des valeurs de `loss` affichées à chaque step, sans `nan`.

In [5]:
!python {SRC}/train_sft.py \
  --train-file {DATA}/train.jsonl \
  --output-dir {OUT}/smoke_test \
  --max-steps 5 \
  --batch-size 1 \
  --gradient-accumulation-steps 1 \
  --logging-steps 1

Chargement du tokenizer et du modele de base : Qwen/Qwen3-1.7B-Base
config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.01MB/s]
tokenizer_config.json: 9.68kB [00:00, 20.3MB/s]
vocab.json: 2.78MB [00:00, 47.9MB/s]
merges.txt: 1.67MB [00:00, 108MB/s]
tokenizer.json: 7.03MB [00:00, 131MB/s]
Précision d'entraînement : fp16
model.safetensors: 100%|████████████████████| 3.44G/3.44G [00:29<00:00, 117MB/s]
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 448kB/s]
Configuration LoRA...
trainable params: 6,422,528 || all params: 1,726,997,504 || trainable%: 0.3719
Chargement du dataset...
Generating train split: 879 examples [00:00, 43447.13 examples/s]
Tokenizing train dataset: 100%|███████| 879/879 [00:01<00:00, 495.33 examples/s]
Building labels for train dataset: 100%|█| 879/879 [00:00<00:00, 1194.72 example
Truncating train dataset: 100%|██████| 879/879 [00:00<00:00, 1508.19 examples/s]
Dropping fully masked examples from train dataset: 100%|█| 87

## 5. Entraînement réel⚠️ À ne lancer qu'après un smoke test réussi.**Lance via *Save Version* > *Save & Run All (Commit)*, pas en exécution interactive** : l'exécution continue en arrière-plan même si tu fermes l'onglet, et c'est indispensable pour un run de plusieurs dizaines de minutes. Les checkpoints sont écrits tous les 50 steps (`--save-steps`).

In [6]:
!python {SRC}/train_sft.py \
  --train-file {DATA}/train.jsonl \
  --val-file {DATA}/validation.jsonl \
  --output-dir {OUT}/qwen3-1.7b-triage-sft \
  --num-epochs 3 \
  --batch-size 2 \
  --gradient-accumulation-steps 8

Chargement du tokenizer et du modele de base : Qwen/Qwen3-1.7B-Base
Précision d'entraînement : fp16
Loading weights: 100%|███████████████████████| 310/310 [00:01<00:00, 160.18it/s]
Configuration LoRA...
trainable params: 6,422,528 || all params: 1,726,997,504 || trainable%: 0.3719
Chargement du dataset...
Generating train split: 879 examples [00:00, 63699.39 examples/s]
Generating validation split: 98 examples [00:00, 20778.58 examples/s]
Tokenizing train dataset: 100%|███████| 879/879 [00:01<00:00, 507.03 examples/s]
Building labels for train dataset: 100%|█| 879/879 [00:00<00:00, 1172.11 example
Truncating train dataset: 100%|██████| 879/879 [00:00<00:00, 1460.18 examples/s]
Dropping fully masked examples from train dataset: 100%|█| 879/879 [00:00<00:00,
Tokenizing eval dataset: 100%|██████████| 98/98 [00:00<00:00, 479.59 examples/s]
Building labels for eval dataset: 100%|█| 98/98 [00:00<00:00, 1117.70 examples/s
Truncating eval dataset: 100%|█████████| 98/98 [00:00<00:00, 1384.33 ex

## 6. Récupérer le résultatSeuls les fichiers de `/kaggle/working` d'une version **sauvegardée** (Save Version) sont conservés. Une fois le commit terminé, l'archive apparaît dans l'onglet *Output* du notebook — télécharge-la de là (adaptateur LoRA + `trainer_state.json`, l'historique des loss, utile pour le rapport).

In [7]:
import shutil

NOM = "qwen3-1.7b-triage-sft"
archive = shutil.make_archive(f"{OUT}/{NOM}", "zip", f"{OUT}/{NOM}")
print("Archive créée :", archive)
print("Après 'Save Version', elle apparaît dans l'onglet Output du notebook — télécharge-la de là.")

Archive créée : /kaggle/working/qwen3-1.7b-triage-sft.zip
Après 'Save Version', elle apparaît dans l'onglet Output du notebook — télécharge-la de là.
